#### Loading all Packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import holidays
from datetime import date
import plotly.express as px
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import BallTree
import requests
import time
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.metrics import mean_squared_error, r2_score
import random
import os
import geopandas as gpd
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.ticker as mticker
import plotly.graph_objects as go
from shapely.geometry import Point
import folium

#### Loading all DataSets

In [ ]:
bikeshare_full = pd.read_csv("CBS_2021-2023_Full.csv", sep=',')
bikeshare_daily = pd.read_csv("CBS_2021-2023_Daily_Weather.csv", sep=',')
bikeshare_hourly = pd.read_csv("CBS_2021-2023_Hourly_Weather.csv", sep=',')

## Merging Datasets on 2023

### Preparation Bikeshare_Full

In [ ]:
# Convert 'date' column to datetime format
bikeshare_full['started_at'] = pd.to_datetime(bikeshare_full['started_at'])

# Keep only rows where the year is 2023
bikeshare_full = bikeshare_full[bikeshare_full['started_at'].dt.year == 2023]

# Create new 'date' column (date only)
bikeshare_full['date'] = bikeshare_full['started_at'].dt.date

# Create new 'month' column (month of the year)
bikeshare_full['month'] = bikeshare_full['started_at'].dt.month

# Create new 'hour' column (hour of the day)
bikeshare_full['hour'] = bikeshare_full['started_at'].dt.hour

# Create new day column (day of the month)
bikeshare_full['day'] = bikeshare_full['started_at'].dt.day

# Create new 'minute' column (minute of the hour)
bikeshare_full['minute'] = bikeshare_full['started_at'].dt.minute

## Drop rows with empty values 
bikeshare_full.dropna(inplace=True)

### Preparation BikeShare_hourly

In [ ]:
## Create Member and Casual Ratio Column (Positive Value = more members than casuals, Negative Value = more casuals than members)

bikeshare_hourly['member/casual_ratio'] = bikeshare_hourly['Member'] / (bikeshare_hourly['Casual'])

## Rename Columns

bikeshare_hourly.rename(columns={'Date': 'date', 'Hour': 'hour', 'Member': 'member', 'Casual': 'casual', 'Total_rides': 'total_rides', 'weathercode (wmo code)': 'weathercode', 'relativehumidity_2m (%)': 'relative_humidity', 'temperature_2m (°C)': 'temperature', 'windspeed_10m (km/h)': 'windspeed'}, inplace=True)

## Delete rows with missing values

bikeshare_hourly.dropna(inplace=True)

## Convert date and hour to datetime

bikeshare_hourly['date'] = pd.to_datetime(bikeshare_hourly['date'])


## Convert temperature to integer
bikeshare_hourly['temperature'] = bikeshare_hourly['temperature'].astype(int)
## Convert total_rides to integer
bikeshare_hourly['total_rides'] = bikeshare_hourly['total_rides'].astype(int)
## Convert member to integer
bikeshare_hourly['member'] = bikeshare_hourly['member'].astype(int)
## Convert casual to integer
bikeshare_hourly['casual'] = bikeshare_hourly['casual'].astype(int)
## Create Year Column
bikeshare_hourly['year'] = bikeshare_hourly['date'].dt.year
## Create Month Column
bikeshare_hourly['month'] = bikeshare_hourly['date'].dt.month
## Create Day Column    
bikeshare_hourly['day'] = bikeshare_hourly['date'].dt.day
## Create minute Column
bikeshare_hourly['minute'] = bikeshare_hourly['date'].dt.minute

## Map each weather code to a numeric group category
def map_weather_group(code):
    if code == 0:
        return 0  # Clear
    elif code in [1, 2, 3]:
        return 1  # Partly Cloudy / Overcast
    elif code in [45, 48]:
        return 2  # Fog
    elif 51 <= code <= 57:
        return 3  # Drizzle
    elif 61 <= code <= 67:
        return 4  # Rain
    elif 71 <= code <= 77:
        return 5  # Snow
    elif 80 <= code <= 82:
        return 6  # Rain Showers
    elif 85 <= code <= 86:
        return 7  # Snow Showers
    elif 95 <= code <= 99:
        return 8  # Thunderstorm
    else:
        return 9  # Unknown / other
bikeshare_hourly['weather_group'] = bikeshare_hourly['weathercode'].apply(map_weather_group)
# Map numeric group to category name
weather_labels = {
    0: 'Clear',
    1: 'Partly Cloudy',
    2: 'Fog',
    3: 'Drizzle',
    4: 'Rain',
    5: 'Snow',
    6: 'Rain Showers',
    7: 'Snow Showers',
    8: 'Thunderstorm',
    9: 'Unknown'
}

bikeshare_hourly['weather_label'] = bikeshare_hourly['weather_group'].map(weather_labels)

## Put Relative Humidity into Categories

def categorize_humidity(humidity):
    if humidity <= 40:
        return 'Low'
    elif humidity <= 70:
        return 'Medium'
    else:
        return 'High'

bikeshare_hourly['humidity_category'] = bikeshare_hourly['relative_humidity'].apply(categorize_humidity)

## Rearrange Column order
bikeshare_hourly = bikeshare_hourly[['date', 'year', 'month', 'day', 'hour', 'total_rides', 'member', 'casual', 'member/casual_ratio', 'weathercode', 'weather_label', 'weather_group', 'temperature', 'windspeed', 'relative_humidity', 'humidity_category']]

### Preparation Bikeshare_Daily

In [ ]:
## Rename Columns
bikeshare_daily.rename(columns={'Date': 'date', 'apparent_temperature_mean (°C)': 'mean_apparent_temperature'}, inplace=True)

## Convert to datetime
bikeshare_daily['date'] = pd.to_datetime(bikeshare_daily['date'])

## Create month column
bikeshare_daily['month'] = bikeshare_daily['date'].dt.month

## Create day of the week column
bikeshare_daily['day'] = bikeshare_daily['date'].dt.day

## Create hour of the day column
bikeshare_daily['hour'] = bikeshare_daily['date'].dt.hour

## Create minute of the hour column
bikeshare_daily['minute'] = bikeshare_daily['date'].dt.minute

## Rearrange Column order
bikeshare_daily = bikeshare_daily[['date', 'month', 'hour', 'minute', 'mean_apparent_temperature']]

### Merge Hourly and Daily Dataset

In [ ]:
# Ensure 'date' column is in datetime format
bikeshare_hourly['date'] = pd.to_datetime(bikeshare_hourly['date'])

# Now safely extract the day
bikeshare_hourly['day'] = bikeshare_hourly['date'].dt.day
bikeshare_hourly['minute'] = bikeshare_hourly['date'].dt.minute

# Ensure 'date' column is in datetime format
bikeshare_daily['date'] = pd.to_datetime(bikeshare_daily['date'])

# Now safely extract the day
bikeshare_daily['day'] = bikeshare_daily['date'].dt.day
bikeshare_daily['minute'] = bikeshare_daily['date'].dt.minute

# Create 'day' column
bikeshare_hourly['day'] = bikeshare_hourly['date'].dt.day

print("bikeshare_hourly columns:", bikeshare_hourly.columns.tolist())
print("bikeshare_daily columns:", bikeshare_daily.columns.tolist())

## Merge both datasets on the basis of the date

# Merge on the 'date' column
bikeshare_merged = pd.merge(bikeshare_hourly, bikeshare_daily, on=['date', 'month', 'day', 'hour', 'minute'], how='left')

## Rearrange Column order
bikeshare_merged = bikeshare_merged[['date', 'month', 'day', 'hour', 'minute', 'total_rides', 'member', 'casual', 'member/casual_ratio', 'weathercode', 'weather_label', 'weather_group', 'temperature', 'mean_apparent_temperature', 'windspeed', 'relative_humidity', 'humidity_category']]

# Keep only rows where the year is 2023
bikeshare_merged2023 = bikeshare_merged[bikeshare_merged['date'].dt.year == 2023]

# Delete rows with missing values
bikeshare_merged2023.dropna(inplace=True)

### Merge Bikeshare_merged2023 and bikeshare_full to arrive at final dataframe for EDA

In [ ]:
# Ensure 'date' column exists and is datetime in both
bikeshare_full['date'] = pd.to_datetime(bikeshare_full['date'])
bikeshare_merged2023['date'] = pd.to_datetime(bikeshare_merged2023['date'])

# Merge on the 'date' column
bikeshare = pd.merge(bikeshare_full, bikeshare_merged2023, on='date', how='left')

# Inspect new merged data frame
bikeshare.head()

## EDA / Feature Engineering

#### Rename Variables

In [ ]:
# Rename variables

bikeshare = bikeshare.rename(columns={'hour_x': 'hour'})
bikeshare['hour'] = bikeshare['hour'].astype(int)

bikeshare = bikeshare.rename(columns={'month_x': 'month'})
bikeshare['month'] = bikeshare['month'].astype(int)

bikeshare = bikeshare.rename(columns={'minute_x': 'minute'})
bikeshare['minute'] = bikeshare['minute'].astype(int)

bikeshare = bikeshare.rename(columns={'member': 'member/day'})
bikeshare['member/day'] = bikeshare['member/day'].astype(int)

bikeshare = bikeshare.rename(columns={'casual': 'casual/day'})
bikeshare['casual/day'] = bikeshare['casual/day'].astype(int)

bikeshare = bikeshare.rename(columns={'total_rides': 'total_rides/day'})
bikeshare['total_rides/day'] = bikeshare['total_rides/day'].astype(int)

bikeshare = bikeshare.rename(columns={'weathercode': 'weathercode/day'})
bikeshare['weathercode/day'] = bikeshare['weathercode/day'].astype(int)

bikeshare = bikeshare.rename(columns={
    'weather_label': 'weather_label/day',
    'weather_group': 'weather_group/day',
    'temperature': 'mean_temperature/day',
    'mean_apparent_temperature': 'mean_apparent_temperature/day',
    'windspeed': 'mean_windspeed/day',
    'relative_humidity': 'mean_relative_humidity/day',
    'humidity_category': 'humidity_category/day',
})

#### Create Rental Length Variable

In [ ]:
# Create Rental Length Variable

bikeshare['started_at'] = pd.to_datetime(bikeshare['started_at'])
bikeshare['ended_at'] = pd.to_datetime(bikeshare['ended_at'])
bikeshare['rental_length'] = bikeshare['ended_at'] - bikeshare['started_at']  
bikeshare['rental_length'] = bikeshare['rental_length'].dt.total_seconds() / 60  
bikeshare['rental_length'] = bikeshare['rental_length'].astype(int)  

#### Create Dummies for Categorical Variables

In [ ]:
# Generate dummy variables but keep original columns
dummies = pd.get_dummies(
    bikeshare[['rideable_type', 'weather_label/day', 'humidity_category/day']],
    prefix=['rideable_type', 'weather_label/day', 'humidity_category/day']
)

# Concatenate dummies with original DataFrame
bikeshare = pd.concat([bikeshare, dummies], axis=1)

### Create Holiday Feature

In [ ]:
# U.S. holidays, specific to Washington, D.C.
dc_holidays = holidays.US(subdiv='DC')

# Ensure 'date' column is datetime
bikeshare['date'] = pd.to_datetime(bikeshare['date'])

# Create boolean column: True if date is a holiday, else False
bikeshare['holiday'] = bikeshare['date'].isin(dc_holidays)

### Create Numeric Variables Counting the Number of Each Rideable Type per Day Normalized by the Total Number of Rides

In [ ]:
# Step 1: Group by 'date' and sum the binary indicators
daily_bike_counts = bikeshare.groupby('date')[[
    'rideable_type_classic_bike',
    'rideable_type_docked_bike',
    'rideable_type_electric_bike'
]].sum().reset_index()

# Step 3: Calculate total rentals per day
daily_bike_counts['total_rentals'] = (
    daily_bike_counts['rideable_type_classic_bike'] +
    daily_bike_counts['rideable_type_docked_bike'] +
    daily_bike_counts['rideable_type_electric_bike']
)

# Step 4: Normalize to get shares
daily_bike_counts['classic_bike_share'] = daily_bike_counts['rideable_type_classic_bike'] / daily_bike_counts['total_rentals']
daily_bike_counts['docked_bike_share'] = daily_bike_counts['rideable_type_docked_bike'] / daily_bike_counts['total_rentals']
daily_bike_counts['electric_bike_share'] = daily_bike_counts['rideable_type_electric_bike'] / daily_bike_counts['total_rentals']

# Step 5: Merge back to bikeshare dataframe
bikeshare = bikeshare.merge(daily_bike_counts[['date', 'classic_bike_share', 'docked_bike_share', 'electric_bike_share']], on='date', how='left')


### Create Numeric Variables Counting the Number of Each Member Type per Day Normalized by the Total Number of Rides

In [ ]:
# Group by month and calculate total rides
monthly_usage = bikeshare.groupby('month')[['member/day', 'casual/day']].sum().reset_index()

# Compute total rides per month
monthly_usage['total_rides'] = monthly_usage['member/day'] + monthly_usage['casual/day']

# Normalize member and casual rides
monthly_usage['member_share'] = monthly_usage['member/day'] / monthly_usage['total_rides']
monthly_usage['casual_share'] = monthly_usage['casual/day'] / monthly_usage['total_rides']

### Create Rental Length in Km Feature

In [ ]:
# Convert degrees to radians
def haversine_vectorized(lat1, lng1, lat2, lng2):
    R = 6371  # Earth radius in kilometers

    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lng2 - lng1)

    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

# Compute distances (vectorized)
bikeshare['trip_distance_km'] = haversine_vectorized(
    bikeshare['start_lat'], bikeshare['start_lng'],
    bikeshare['end_lat'], bikeshare['end_lng']
)


### Add Seasonality Feature

In [ ]:
def date_to_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:  # [9, 10, 11]
        return 'Autumn'

bikeshare['season'] = bikeshare['date'].apply(date_to_season)


### Create Weekend Column

In [ ]:
# Ensure 'date' column is in datetime format
bikeshare['date'] = pd.to_datetime(bikeshare['date'])

# Create 'is_weekend' column: True for Saturday (5) or Sunday (6), else False
bikeshare['is_weekend'] = bikeshare['date'].dt.weekday >= 5


### Hot Encode Season, weather_label/day

In [ ]:
# One-hot encode the 'season' variable and concatenate with the original DataFrame, then drop the old 'season' column

season_dummies = pd.get_dummies(bikeshare['season'], prefix='season')
bikeshare = pd.concat([bikeshare, season_dummies], axis=1)

weather_label_dummies = pd.get_dummies(bikeshare['weather_label/day'], prefix='weather_label')
bikeshare = pd.concat([bikeshare, weather_label_dummies], axis=1)

### Create Lag-Feature Total_Rides_Yesterday

In [ ]:
# Create a new column: yesterday's total rides
bikeshare['total_rides_yesterday'] = bikeshare['total_rides/day'].shift(1)
bikeshare.dropna(subset=['total_rides_yesterday'])

### Add day of the week feature

In [ ]:
# Add day of week feature (0=Monday, 6=Sunday)
bikeshare['day_of_week'] = bikeshare['date'].dt.dayofweek

### Add day feature

In [ ]:
# Ensure 'date' column is in datetime format
bikeshare['date'] = pd.to_datetime(bikeshare['date'])

# Create 'day' column (day of the month, 1-31)
bikeshare['day'] = bikeshare['date'].dt.day

## ML Model, Feature Engineering and Visualization

In [ ]:
# import numpy as np
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
# from scipy import stats
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.neighbors import BallTree
# import requests
# import time
# import requests
# import time

### ML Features/ML Dataframe

In [ ]:
### STEP 1: Aggregate bikeshare data to station-day level ###
start_station_daily = (
    bikeshare
    .groupby(['start_station_id', 'date'])
    .agg({
        'ride_id': 'count', 
        'member/day': 'sum',
        'casual/day': 'sum',
        'rideable_type_classic_bike': 'sum',
        'rideable_type_docked_bike': 'sum',
        'rideable_type_electric_bike': 'sum',
        'rental_length': 'mean',
        'trip_distance_km': 'mean',
        'mean_temperature/day': 'first',
        'mean_apparent_temperature/day': 'first',
        'mean_windspeed/day': 'first',
        'mean_relative_humidity/day': 'first',
        'weather_label/day': 'first',
        'weather_label_Clear': 'first',
        'weather_label_Drizzle': 'first',
        'weather_label_Partly Cloudy': 'first',
        'weather_label_Rain': 'first',
        'weather_label_Snow': 'first',
        'season_Summer': 'first',
        'season_Winter': 'first',
        'season_Spring': 'first',
        'season_Autumn': 'first',
        'season': 'first',
        'is_weekend': 'first',
        'holiday': 'first',
        'total_rides/day': 'first',
        'month': 'first',
        'day': 'first',
        'start_lat': 'first',
        'start_lng': 'first',
        'day_of_week': 'first'
    })
    .reset_index()
    .rename(columns={'ride_id': 'total_rides/day_station'})
)

### STEP 2: Add lag feature ###
start_station_daily = start_station_daily.sort_values(['start_station_id', 'date'])
start_station_daily['total_rides/day_station_yesterday'] = (
    start_station_daily
    .groupby('start_station_id')['total_rides/day_station']
    .shift(1)
)
start_station_daily = start_station_daily.dropna(subset=['total_rides/day_station_yesterday'])

### STEP 2: Add lag feature ###
start_station_daily = start_station_daily.sort_values(['start_station_id', 'date'])
start_station_daily['total_rides/day_station_yesterday'] = (
    start_station_daily
    .groupby('start_station_id')['total_rides/day_station']
    .shift(1)
)
# Add lag feature for 7 days ago
start_station_daily['total_rides/day_station_weekago'] = (
    start_station_daily
    .groupby('start_station_id')['total_rides/day_station']
    .shift(7)
)
# Drop rows with missing values in either lag feature
start_station_daily = start_station_daily.dropna(
    subset=['total_rides/day_station_yesterday', 'total_rides/day_station_weekago']
)

### STEP 3: Add average usage ###
start_station_daily['avg_usage'] = (
    start_station_daily
    .groupby('start_station_id')['total_rides/day_station']
    .transform('mean')
)

### STEP 4: Add popularity rank ###
station_usage = (
    start_station_daily
    .groupby('start_station_id')['total_rides/day_station']
    .sum()
    .reset_index()
)
station_usage['popularity_rank'] = (
    station_usage['total_rides/day_station']
    .rank(method='dense', ascending=False)
    .astype(int)
)

assert station_usage['start_station_id'].is_unique
start_station_daily = start_station_daily.merge(
    station_usage[['start_station_id', 'popularity_rank']],
    on='start_station_id',
    how='left'
)

### STEP 5: Add station density (within 1 km) ###
# Get one coordinate pair per station
unique_stations = (
    start_station_daily
    .groupby('start_station_id')[['start_lat', 'start_lng']]
    .first()
    .reset_index()
)

# Use BallTree to calculate neighbors
coords = np.radians(unique_stations[['start_lat', 'start_lng']].values)
tree = BallTree(coords, metric='haversine')
radius = 1.0 / 6371  # 1 km in radians

density = tree.query_radius(coords, r=radius, count_only=True)
station_density = pd.DataFrame({
    'start_station_id': unique_stations['start_station_id'].values,
    'station_density_1km': density
})

assert station_density['start_station_id'].is_unique
start_station_daily = start_station_daily.merge(
    station_density,
    on='start_station_id',
    how='left'
)

### STEP 7: Final integrity check ###
duplicates = start_station_daily.duplicated(subset=['start_station_id', 'date'])
assert duplicates.sum() == 0, "Duplicate rows found after merging!"

### STEP 8: Sanity check for expected features ###
expected_cols = [
    'avg_usage', 'popularity_rank', 'station_density_1km',
    'total_rides/day_station_yesterday'
]

### STEP 9: Ward Variable

station_id_ward = pd.read_csv('station_id_ward.csv')

# Keep only the first ward per station (if duplicates exist)
station_id_ward_unique = station_id_ward.drop_duplicates(subset=['start_station_id'])

# Now merge with the unique mapping
start_station_daily = start_station_daily.merge(
    station_id_ward_unique[['start_station_id', 'ward']],
    on='start_station_id',
    how='left'
)

# Load the station_id_ward mapping from CSV
station_id_ward = pd.read_csv('station_id_ward.csv')

ward_check = station_id_ward.groupby('start_station_id')['ward'].nunique()

# Use first ward per station
station_id_ward_unique = station_id_ward.drop_duplicates(subset=['start_station_id'], keep='first')
assert station_id_ward_unique['start_station_id'].is_unique

start_station_daily = start_station_daily.merge(
    station_id_ward_unique[['start_station_id', 'ward']],
    on='start_station_id',
    how='left'
)

# Remove duplicate rows from station_id_ward
station_id_ward = station_id_ward.drop_duplicates()

# Identify stations assigned to multiple wards
multi_ward_ids = (
    station_id_ward.groupby('start_station_id')['ward']
    .nunique()
    .reset_index()
    .query('ward > 1')['start_station_id']
)

# Drop all rows for stations assigned to multiple wards
station_id_ward_clean = station_id_ward[~station_id_ward['start_station_id'].isin(multi_ward_ids)].copy()

# Optionally, overwrite the original variable
station_id_ward = station_id_ward_clean

# Merge cleaned station_id_ward (one ward per station) onto start_station_daily
start_station_daily = start_station_daily.merge(
    station_id_ward[['start_station_id', 'ward']],
    on='start_station_id',
    how='left'
)

### STEP 10: Get Elevation Variable

# Get unique station coordinates
unique_stations = start_station_daily.groupby('start_station_id')[['start_lat', 'start_lng']].first().reset_index()

# Prepare to store elevation results
elevations = []

for idx, row in unique_stations.iterrows():
    lat, lng = row['start_lat'], row['start_lng']
    url = f"https://api.open-elevation.com/api/v1/lookup?locations={lat},{lng}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        if 'results' in data and len(data['results']) > 0:
            elevation = data['results'][0]['elevation']
        else:
            elevation = None
    except Exception as e:
        elevation = None
    elevations.append(elevation)
    time.sleep(0.1)  # Be polite to the API

# Add elevation to the DataFrame
unique_stations['elevation'] = elevations

# Merge elevation back into start_station_daily
start_station_daily = start_station_daily.merge(
    unique_stations[['start_station_id', 'elevation']],
    on='start_station_id',
    how='left'
)

### STEP 11: Add Hour Variable 
# Add 'hour' column to start_station_daily by taking the first hour of the day for each station-date

hour_lookup = (
    bikeshare.groupby(['start_station_id', 'date'])['hour']
    .first()
    .reset_index()
)

start_station_daily = start_station_daily.merge(
    hour_lookup,
    on=['start_station_id', 'date'],
    how='left'
)

### STEP 12: Convert day and day_of_week to categorical dtype
bikeshare['day'] = bikeshare['day'].astype('category')
bikeshare['day_of_week'] = bikeshare['day_of_week'].astype('category')


### ML Splitting

In [ ]:
X = start_station_daily[[
    'start_lat',
    'start_lng',
    'start_station_id',
    'day',
    'month',
    'rental_length', 
    'trip_distance_km', 
    'mean_apparent_temperature/day',
    'mean_windspeed/day', 
    'mean_relative_humidity/day', 
    'weather_label/day',
    'season',
    'is_weekend', 
    'holiday',
    'total_rides/day_station_yesterday',
    'day_of_week',
    'avg_usage',
    'station_density_1km',
    'popularity_rank',
    'ward',
    'total_rides/day_station_weekago',
    'elevation',
]]

y = start_station_daily['total_rides/day_station']

### ML Model Pipeline

In [ ]:
# from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
# from xgboost import XGBRegressor
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler, OrdinalEncoder
# from sklearn.compose import ColumnTransformer, make_column_selector as selector
# from sklearn.metrics import mean_squared_error, r2_score
# import random
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

In [ ]:

np.random.seed(42)
random.seed(42)
os.environ['PYTHONHASHSEED'] = '42'

tscv = TimeSeriesSplit(n_splits=4)
all_results = []
rmse_scores = []
r2_scores = []

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Define transformers
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline(steps=[
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, selector(dtype_include=['int64', 'float64'])),
            ('cat', categorical_transformer, selector(dtype_include=['object', 'category']))
        ]
    )

    # Create pipeline with XGBRegressor
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', XGBRegressor(
            objective='reg:squarederror',
            random_state=42,
            n_jobs=1,
            verbosity=0
        ))
    ])

    # Grid Search Parameter Grid
    param_grid = {
        'regressor__learning_rate': [0.06],
        'regressor__max_depth': [12],
        'regressor__n_estimators': [360],
        'regressor__subsample': [0.64],
        'regressor__colsample_bytree': [0.96]
    }

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=2,
        scoring='neg_mean_squared_error',
        verbose=0,
        n_jobs=1
    )

    # Fit model
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    # Feature Importance
    importances = best_model.named_steps['regressor'].feature_importances_
    feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

    feat_imp_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values(by='importance', ascending=False)

    # # Optional: Plot
    # plt.figure(figsize=(8, 6))
    # sns.barplot(data=feat_imp_df.head(23), x='importance', y='feature', palette='viridis')
    # plt.title('Top 10 Most Important Features')
    # plt.tight_layout()
    # plt.show()

    # Predict on test set
    y_pred = best_model.predict(X_test)

    # Print GridSearchCV results for this split
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    print(f"Split RMSE: {rmse:.2f}")
    print(f"Split R²: {r2:.2f}")
    print("Best hyperparameters found:", grid_search.best_params_)

    # Collect results for this split
    temp_df = pd.DataFrame({
        'date': start_station_daily.loc[y_test.index, 'date'],
        'actual': y_test,
        'predicted': y_pred,
        'residual': y_test - y_pred
    })
    all_results.append(temp_df)

    # Collect scores for averaging
    rmse_scores.append(rmse)
    r2_scores.append(r2)

# After the loop, concatenate all results
results_df = pd.concat(all_results, ignore_index=True)

# Print average results
print(f"Average RMSE across splits: {np.mean(rmse_scores):.3f}")
print(f"Average R² across splits: {np.mean(r2_scores):.3f}")

### Best Features in ML Model

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

In [ ]:
# Select top N features
TOP_N = 20
top_features = feat_imp_df.sort_values('importance', ascending=False).head(TOP_N).copy()

# Feature renaming
feature_rename = {
    'num__start_lat': 'Latitude',
    'num__start_lng': 'Longitude',
    'num__rental_length': 'Rental Length',
    'num__trip_distance_km': 'Trip Distance (km)',
    'num__mean_apparent_temperature/day': 'Mean Apparent Temp',
    'num__mean_windspeed/day': 'Mean Windspeed',
    'num__mean_relative_humidity/day': 'Mean Rel. Humidity',
    'num__total_rides/day_station_yesterday': 'Rides Yesterday',
    'num__total_rides/day_station_weekago': 'Rides Week Ago',
    'num__avg_usage': 'Avg Usage',
    'num__station_density_1km': 'Station Density (1km)',
    'num__popularity_rank': 'Popularity Rank',
    'num__elevation': 'Elevation',
    'cat__season': 'Season',
    'cat__is_weekend': 'Is Weekend?',
    'cat__holiday': 'Holiday',
    'cat__day_of_week': 'Day of Week',
    'cat__weather_label/day': 'Weather Label',
    'num__month': 'Month',
    'cat__ward': 'Ward',
    'cat__day': 'Day',
    'num__ward': 'Ward',
    'num__start_station_id': 'Start Station ID'
}
top_features['feature_label'] = top_features['feature'].map(feature_rename).fillna(top_features['feature'])

# Plot
plt.figure(figsize=(10, 7))
barplot = sns.barplot(
    data=top_features,
    x='importance',
    y='feature_label',
    color='#df9c20'
)

# Add value labels in grey
for p in barplot.patches:
    width = p.get_width()
    y_pos = p.get_y() + p.get_height() / 2
    barplot.annotate(f"{width:.2f}",
                     (width + 0.005, y_pos),
                     va='center',
                     ha='left',
                     fontsize=9,
                     color='grey')

# Styling: feature names and x-label in black, all other text in grey, no header
plt.xlabel('Importance', fontsize=12, color='black')
plt.ylabel('')
plt.xticks(color='grey')
plt.yticks(color='black')  # Feature names in black
plt.grid(axis='x', linestyle='--', alpha=0.5)
sns.despine()
plt.tight_layout()
plt.savefig("most_important_features_orange.png", dpi=300, transparent=True)
plt.show()

### Analysis of Residuals

In [ ]:
# Fit on all data
best_model.fit(X, y)
y_pred_full = best_model.predict(X)

# Create a DataFrame for all dates, including station id
results_full_df = pd.DataFrame({
    'date': start_station_daily['date'],
    'start_station_id': start_station_daily['start_station_id'],  # <-- add this line
    'actual': y,
    'predicted': y_pred_full,
    'residual': y - y_pred_full
})

# Now use results_full_df for outlier detection
threshold = 5
outliers_df = results_full_df[np.abs(results_full_df['residual']) > threshold]
outliers_df = outliers_df.sort_values(by='residual', key=np.abs, ascending=False)

# Keep the most extreme residual per date (and station)
unique_events_df = outliers_df.loc[outliers_df.groupby(['date'])['residual'].apply(lambda x: np.abs(x).idxmax())]
unique_events_df = unique_events_df.reindex(unique_events_df['residual'].abs().sort_values(ascending=False).index)
unique_events_df = unique_events_df.reset_index(drop=True)

# Now print including station id
print(unique_events_df[['date', 'start_station_id', 'actual', 'predicted', 'residual']])

# Create a DataFrame for all dates
results_full_df = pd.DataFrame({
    'date': pd.to_datetime(start_station_daily['date']),
    'actual': y,
    'predicted': y_pred_full,
    'residual': y - y_pred_full
})

plt.figure(figsize=(14, 5))
plt.scatter(
    results_full_df['date'],
    results_full_df['residual'],
    alpha=0.6,
    s=20,
    c='#c79538',   # Use the requested color
    edgecolor='k'
)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.axhline(5, color='red', linestyle=':', linewidth=1)   # Thin red line at +5
plt.axhline(-5, color='red', linestyle=':', linewidth=1)  # Thin red line at -5
plt.title('Distribution of Residuals (Station Level Prediction Whole Year)')
plt.xlabel('Date')
plt.ylabel('Residual (Actual - Predicted)')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()

# Save the figure as PNG
plt.savefig("residuals_station_level_whole_year.png", dpi=300, transparent=True)

plt.show()

within_5 = (results_full_df['residual'].abs() <= 5).mean() * 100
print(f"Percentage of residuals within ±5: {within_5:.2f}%")

### Interactive Plot of all Residuals outside of 95% Confidence Interval (with Indication of Station ID)

In [ ]:
fig = px.scatter(
    unique_events_df,
    x='date',
    y='residual',
    color=unique_events_df['residual'].abs(),
    size=unique_events_df['residual'].abs(),
    custom_data=['date', 'start_station_id', 'actual', 'predicted', 'residual'],
    labels={
        'residual': 'Residual',
        'date': 'Date',
        'start_station_id': 'Station ID'
    },
    title='Special Event Outliers: Residuals Over the Year'
)

fig.update_traces(
    hovertemplate=
        "Date: %{customdata[0]}<br>" +
        "Station ID: %{customdata[1]}<br>" +
        "Actual: %{customdata[2]}<br>" +
        "Predicted: %{customdata[3]}<br>" +
        "Residual: %{customdata[4]}<extra></extra>"
)

fig.update_layout(
    width=1400,
    height=500,
    margin=dict(l=40, r=40, t=80, b=40),
    title={
        'text': 'ML Residuals Over the Year',
        'x': 0.5,
        'xanchor': 'center',
        'font': dict(size=18)
    },
    font=dict(size=10)
)

fig.update_traces(marker=dict(line=dict(width=1, color='black')))
fig.update_layout(coloraxis_colorbar=dict(title='Residual', title_font=dict(size=12), tickfont=dict(size=10)))
fig.write_html("ML_residuals_over_the_year.html")
fig.show()

### Residuals vs. Prediction

In [ ]:
# Create a DataFrame for all dates
results_full_df = pd.DataFrame({
    'actual': y,
    'predicted': y_pred_full,
    'residual': y - y_pred_full
})

# Scatter plot: Actual vs Predicted (full model)
plt.figure(figsize=(6, 6))
plt.scatter(results_full_df['actual'], results_full_df['predicted'], alpha=0.5)
plt.plot(
    [results_full_df['actual'].min(), results_full_df['actual'].max()],
    [results_full_df['actual'].min(), results_full_df['actual'].max()],
    '--r'
)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Predicted vs Actual Ride Volume (Full Model)')
plt.tight_layout()
plt.show()

### Numeric Analysis of Prediction Results

In [ ]:
# Count number of stations exceeding the threshold for each day
stations_per_day = outliers_df.groupby('date')['start_station_id'].nunique()

# Calculate the average per day
average_stations_per_day = stations_per_day.mean()
print(f"Average number of stations exceeding the threshold per day: {average_stations_per_day:.2f}")

# Calculate the total number of unique stations that ever exceed the threshold in the year
total_unique_stations = outliers_df['start_station_id'].nunique()
print(f"Total number of unique stations exceeding the threshold in the year: {total_unique_stations}")

# Calculate the total number of station-day outlier events in the year (sum of stations per day)
total_station_day_events = stations_per_day.sum()
print(f"Total number of station-day outlier events in the year: {total_station_day_events}")

## Geocoding

### Filter Stations that fall within Ward

In [ ]:
### Add Ward Polygons for Geospatial Mapping
# import geopandas as gpd
# import pandas as pd

# Load ward polygons from GeoJSON (this preserves geometry)
wards_gdf = gpd.read_file("Wards_from_2022.geojson")
wards_gdf = wards_gdf.to_crs(epsg=4326)  # Ensure CRS is WGS84

# Create GeoDataFrame for stations
stations_gdf = gpd.GeoDataFrame(
    start_station_daily,
    geometry=gpd.points_from_xy(start_station_daily['start_lng'], start_station_daily['start_lat']),
    crs="EPSG:4326"
)

# Spatial join: keep only stations within any ward polygon
stations_in_wards = gpd.sjoin(stations_gdf, wards_gdf, how='inner', predicate='within')

# Drop geometry columns if you want a regular DataFrame
stations_in_wards = stations_in_wards.drop(columns=['geometry', 'index_right'])

### Assign each station ID to a Ward

In [ ]:
# import geopandas as gpd

# Load the wards GeoDataFrame
wards = gpd.read_file("Wards_from_2022.geojson")
wards = wards[['geometry', 'WARD']]

# Create a DataFrame with unique station IDs and their coordinates
station_coords = bikeshare[['start_station_id', 'start_lat', 'start_lng']].drop_duplicates()

# Convert to GeoDataFrame
station_gdf = gpd.GeoDataFrame(
    station_coords,
    geometry=gpd.points_from_xy(station_coords['start_lng'], station_coords['start_lat']),
    crs="EPSG:4326"
)

# Spatial join to assign each station to a ward
station_gdf = gpd.sjoin(station_gdf, wards, how="left", predicate="within")

# Keep only station_id and ward columns
station_id_ward = station_gdf[['start_station_id', 'WARD']].rename(columns={'WARD': 'ward'})

# Remove rows with missing ward values in station_id_ward
station_id_ward = station_id_ward.dropna(subset=['ward'])

### Add Start and End Ward to each trip

In [ ]:
# Create GeoDataFrame for start coordinates
bikeshare_start = bikeshare.copy()
bikeshare_start['geometry'] = gpd.points_from_xy(bikeshare_start['start_lng'], bikeshare_start['start_lat'])
bikeshare_start_gdf = gpd.GeoDataFrame(bikeshare_start, geometry='geometry', crs="EPSG:4326")

# Spatial join for start coordinates
start_join = gpd.sjoin(bikeshare_start_gdf, wards, how="left", predicate="within")
bikeshare['start_ward'] = start_join['WARD']

# Create GeoDataFrame for end coordinates
bikeshare_end = bikeshare.copy()
bikeshare_end['geometry'] = gpd.points_from_xy(bikeshare_end['end_lng'], bikeshare_end['end_lat'])
bikeshare_end_gdf = gpd.GeoDataFrame(bikeshare_end, geometry='geometry', crs="EPSG:4326")

# Spatial join for end coordinates
end_join = gpd.sjoin(bikeshare_end_gdf, wards, how="left", predicate="within")
bikeshare['end_ward'] = end_join['WARD']

## Geospatial Mapping

### Dataframe for Start Wards, Geometry and Total Rides

In [ ]:
# import numpy as np

# # Group by start_ward and sum total_rides/day
# rides_per_start_ward = bikeshare.groupby('start_ward', as_index=False)['total_rides/day'].sum()

# # Remove non-finite values from start_ward in both DataFrames
# rides_per_start_ward = rides_per_start_ward[
#     rides_per_start_ward['start_ward'].apply(lambda x: pd.notnull(x) and np.isfinite(float(x)))
# ]
# wards_renamed = wards.rename(columns={'WARD': 'start_ward'})
# wards_renamed = wards_renamed[
#     wards_renamed['start_ward'].apply(lambda x: pd.notnull(x) and np.isfinite(float(x)))
# ]

# # Convert start_ward to int in both DataFrames
# rides_per_start_ward['start_ward'] = rides_per_start_ward['start_ward'].astype(float).astype(int)
# wards_renamed['start_ward'] = wards_renamed['start_ward'].astype(float).astype(int)

# # Merge with wards to get geometry
# start_ward_geometry = rides_per_start_ward.merge(
#     wards_renamed,
#     on='start_ward',
#     how='left'
# )[['start_ward', 'geometry', 'total_rides/day']]

# start_ward_geometry = gpd.GeoDataFrame(start_ward_geometry, geometry='geometry', crs=wards.crs)

### Net Interward Bike Flows Map

### Geospatial summaries of bikeshare usage per Ward (Where do trips start and where do they end?)

In [ ]:
# Group by start_ward and sum total_rides/day
rides_per_start_ward = bikeshare.groupby('start_ward', as_index=False)['total_rides/day'].sum()

# Merge with wards to get geometry
start_ward_geometry = rides_per_start_ward.merge(
    wards.rename(columns={'WARD': 'start_ward'}),
    on='start_ward',
    how='left'
)[['start_ward', 'geometry', 'total_rides/day']]

start_ward_geometry = gpd.GeoDataFrame(start_ward_geometry, geometry='geometry', crs=wards.crs)

# Ensure both columns are string and stripped
bikeshare['end_ward'] = bikeshare['end_ward'].astype(str).str.strip()
wards['WARD'] = wards['WARD'].astype(str).str.strip()

# Group by start_ward and sum total_rides/day
rides_per_end_ward = bikeshare.groupby('end_ward', as_index=False)['total_rides/day'].sum()

# Merge with wards to get geometry
end_ward_geometry = rides_per_end_ward.merge(
    wards.rename(columns={'WARD': 'end_ward'}),
    on='end_ward',
    how='left'
)[['end_ward', 'geometry', 'total_rides/day']]

end_ward_geometry = gpd.GeoDataFrame(end_ward_geometry, geometry='geometry', crs=wards.crs)

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.lines as mlines
# import matplotlib.colors as mcolors
# import matplotlib.cm as cm
# import matplotlib.ticker as mticker
# import numpy as np

# 1. Prepare data

def calculate_net_flows(df):
    forward = df.groupby(['start_ward', 'end_ward'])['count'].sum().reset_index()
    reverse = df.groupby(['end_ward', 'start_ward'])['count'].sum().reset_index()
    reverse.columns = ['start_ward', 'end_ward', 'reverse_count']
    merged = pd.merge(forward, reverse, on=['start_ward', 'end_ward'], how='outer').fillna(0)
    merged['net_flow'] = merged['count'] - merged['reverse_count']
    net_flows = merged[merged['net_flow'] != 0].copy()
    net_flows['magnitude'] = net_flows['net_flow'].abs()
    # Ensure net direction goes from higher to lower
    net_flows[['start_ward', 'end_ward']] = net_flows.apply(
        lambda row: pd.Series([row['start_ward'], row['end_ward']])
        if row['net_flow'] > 0 else pd.Series([row['end_ward'], row['start_ward']]),
        axis=1
    )
    return net_flows[['start_ward', 'end_ward', 'magnitude']]

# Make sure ward IDs are strings
bikeshare['start_ward'] = bikeshare['start_ward'].astype(str)
bikeshare['end_ward'] = bikeshare['end_ward'].astype(str)

# Create flow tables
classic_flows = (
    bikeshare[bikeshare['rideable_type_classic_bike'] == True]
    .groupby(['start_ward', 'end_ward'])
    .size().reset_index(name='count')
)
electric_flows = (
    bikeshare[bikeshare['rideable_type_electric_bike'] == True]
    .groupby(['start_ward', 'end_ward'])
    .size().reset_index(name='count')
)

# Calculate net flows
classic_net = calculate_net_flows(classic_flows)
electric_net = calculate_net_flows(electric_flows)

# Merge both into one table
combined_net = pd.merge(classic_net, electric_net, on=['start_ward', 'end_ward'], how='outer', suffixes=('_classic', '_electric')).fillna(0)
combined_net['total'] = combined_net['magnitude_classic'] + combined_net['magnitude_electric']
combined_net['dominant'] = np.where(
    combined_net['magnitude_classic'] >= combined_net['magnitude_electric'], 'classic', 'electric'
)

# 2. Prepare centroids

start_ward_geometry = start_ward_geometry.drop_duplicates(subset='start_ward')
ward_centroids = start_ward_geometry.set_index("start_ward").geometry.centroid
ward_centroids.index = ward_centroids.index.astype(str)

# 3. Plot map

fig, ax = plt.subplots(figsize=(12, 12), dpi=300)
ax.axis("off")

# Background map
cmap = "YlOrBr"
start_ward_geometry.plot(
    ax=ax,
    column="total_rides/day",
    cmap=cmap,
    edgecolor="white",
    linewidth=0.3,
    alpha=0.9
)

# Add a thin black outline around all wards
start_ward_geometry.boundary.plot(ax=ax, color='grey', linewidth=0.7, zorder=3)

# Add ward labels
for idx, row in start_ward_geometry.iterrows():
    centroid = row['geometry'].centroid
    ward_label = row['start_ward']
    ax.text(
        centroid.x, centroid.y, str(ward_label),
        fontsize=10, fontweight='bold', color='black',
        ha='center', va='center',
        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', boxstyle='round,pad=0.2')
    )

# 4. Draw net flow arrows

color_map = {'classic': 'tab:blue', 'electric': 'tab:orange'}
threshold = 500  # Minimum flow to draw
min_width = 1
max_width = 6
max_total = combined_net['total'].max()
drawn = 0

for _, row in combined_net.iterrows():
    if row['total'] < threshold:
        continue
    start = ward_centroids.get(row['start_ward'])
    end = ward_centroids.get(row['end_ward'])
    if start is None or end is None:
        continue
    width = min_width + (max_width - min_width) * (row['total'] / max_total)
    color = color_map[row['dominant']]
    ax.annotate(
        '',
        xy=(end.x, end.y),
        xytext=(start.x, start.y),
        arrowprops=dict(
            arrowstyle='->',
            color=color,
            lw=width,
            alpha=0.75,
            shrinkA=2, shrinkB=2,
            mutation_scale=10 + width * 2,
        )
    )
    drawn += 1

# 5. Add colorbar & legend

norm = mcolors.Normalize(vmin=start_ward_geometry["total_rides/day"].min(),
                         vmax=start_ward_geometry["total_rides/day"].max())
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
# cbar = fig.colorbar(sm, ax=ax, shrink=0.3)
# cbar.set_label("Total Rides per Day", fontsize=10)
# cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
# cbar.outline.set_edgecolor("lightgrey")
# cbar.outline.set_linewidth(0.2)


# # Centered main title (large header)
# ax.set_title(
#     "Net Bike Flows Between Wards (Classic & Electric)",
#     fontsize=16,
#     loc='center',
#     weight='bold',
#     pad=30  # Add some space below the title
# )

# Centered sub-header (smaller, grey, just below the main title)
ax.text(
    1.0, 0.01,
    f"Arrows show net flow direction • Color = dominant bike type • Width = volume (> {threshold})",
    transform=ax.transAxes,
    fontsize=9,
    color='gray',
    verticalalignment='bottom',
    horizontalalignment='right'
)


# Manual legend (larger and top-right)
blue_line = mlines.Line2D([], [], color='tab:blue', lw=4, label='Classic-Dominant Flow')
orange_line = mlines.Line2D([], [], color='tab:orange', lw=4, label='Electric-Dominant Flow')

ax.legend(
    handles=[blue_line, orange_line],
    loc='upper right',
    bbox_to_anchor=(1.0, 0.9),
    fontsize=12,
    title='Flow Type',
    title_fontsize=12,
    frameon=False,
    framealpha=0.9,
    facecolor='white',
    edgecolor='gray'
)

plt.tight_layout()
plt.savefig("net_bike_flows.png", dpi=300, bbox_inches='tight', transparent=True)
plt.show()


### Net Interward BikeFlow BarPlot

In [ ]:
# import matplotlib.pyplot as plt

# Calculate net inflow per ward (total incoming - total outgoing)
net_flow = pd.DataFrame({
    'net_inflow': (
        combined_net.groupby('end_ward')['total'].sum() - 
        combined_net.groupby('start_ward')['total'].sum()
    )
}).fillna(0)
net_flow.index.name = 'ward'

# Now you can safely use net_flow
net_flow_clean = net_flow[net_flow.index.notna()].copy()
# ...rest of your plotting code...

# Remove NaN index and any 'nan' string from the index
net_flow_clean = net_flow[net_flow.index.notna()].copy()
net_flow_clean.index = net_flow_clean.index.map(lambda x: str(x).replace('.0', '') if '.0' in str(x) else str(x))
net_flow_clean = net_flow_clean[net_flow_clean.index != 'nan']

# Reverse the sign of all net inflow values
net_flow_clean['net_inflow'] = -net_flow_clean['net_inflow']

# Sort for plotting
net_flow_sorted = net_flow_clean.sort_values('net_inflow', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
bars = ax.bar(
    net_flow_sorted.index,  # Ward numbers as strings, no '.0' or 'nan'
    net_flow_sorted['net_inflow'],
    color='#df9c20'
)

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_xlabel("Ward")
ax.set_ylabel("Net Bike Flow")
ax.set_title("Net Inflow/Outflow per Ward", fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()

# Add vertical and horizontal grid
ax.grid(axis='y', linestyle=':', color='gray', alpha=0.6)
ax.grid(axis='x', linestyle=':', color='gray', alpha=0.4)

# Despine (remove top and right spines)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Save with transparent background
plt.savefig("net_inflow_per_ward_reversed.png", dpi=300, bbox_inches='tight', transparent=True)

### Sankey Diagramm for Interward Bike Flow

In [ ]:
# import plotly.graph_objects as go

In [ ]:
# --- STEP 1.0 Clean up ward columns in bikeshare ---
bikeshare = bikeshare.copy()
bikeshare = bikeshare[
    bikeshare['start_ward'].notna() &
    bikeshare['end_ward'].notna() &
    (bikeshare['start_ward'].astype(str).str.lower() != 'nan') &
    (bikeshare['end_ward'].astype(str).str.lower() != 'nan')
]
bikeshare['start_ward'] = bikeshare['start_ward'].apply(lambda x: int(float(x)))
bikeshare['end_ward'] = bikeshare['end_ward'].apply(lambda x: int(float(x)))

# --- STEP 1.5: Build ward_flows from your bikeshare DataFrame ---

ward_flows = {}
for (start, end), count in bikeshare.groupby(['start_ward', 'end_ward']).size().items():
    # Skip if start or end is NaN or the string 'nan'
    if pd.isna(start) or pd.isna(end) or str(start).lower() == 'nan' or str(end).lower() == 'nan':
        continue
    start = int(float(start))
    end = int(float(end))
    if start not in ward_flows:
        ward_flows[start] = {}
    ward_flows[start][end] = count

ward_ids = sorted(ward_flows.keys())

# --- STEP 2: Define colors for each ward ---
ward_colors = {
    1: 'rgba(31, 119, 180, 0.8)',
    2: 'rgba(255, 127, 14, 0.8)',
    3: 'rgba(44, 160, 44, 0.8)',
    4: 'rgba(214, 39, 40, 0.8)',
    5: 'rgba(148, 103, 189, 0.8)',
    6: 'rgba(140, 86, 75, 0.8)',
    7: 'rgba(227, 119, 194, 0.8)',
    8: 'rgba(127, 127, 127, 0.8)'
}

# --- STEP 3: Create node labels ---
left_labels = [f"Ward {i} (From)" for i in ward_ids]
right_labels = [f"Ward {i} (To)" for i in ward_ids]
labels = left_labels + right_labels

# --- STEP 4: Build link data ---
sources, targets, values, link_colors, link_customdata = [], [], [], [], []

for from_ward in ward_ids:
    for to_ward in ward_ids:
        count = ward_flows[from_ward].get(to_ward, 0)
        if count == 0:
            continue
        src_idx = ward_ids.index(from_ward)
        tgt_idx = len(ward_ids) + ward_ids.index(to_ward)
        sources.append(src_idx)
        targets.append(tgt_idx)
        values.append(count)
        rgba = ward_colors.get(from_ward, 'rgba(200,200,200,0.8)')
        link_colors.append(rgba.replace("0.8", "0.4"))
        link_customdata.append((from_ward, to_ward, count))

# --- STEP 5: Assign node colors ---
node_colors = [ward_colors.get(i, 'rgba(200,200,200,0.8)') for i in ward_ids] * 2

# --- STEP 6: Calculate outgoing/incoming + per-ward ride shares ---
total_flows = sum(values)
outgoing = {ward: sum(ward_flows[ward].get(w, 0) for w in ward_ids) for ward in ward_ids}
incoming = {ward: sum(ward_flows[w].get(ward, 0) for w in ward_ids) for ward in ward_ids}
customdata = []

# Left nodes: "Ward X (From)" - use rides starting in this ward
for ward in ward_ids:
    out_abs = outgoing[ward]
    out_rel = out_abs / total_flows * 100 if total_flows else 0
    in_abs = incoming[ward]
    in_rel = in_abs / total_flows * 100 if total_flows else 0
    ward_rides = bikeshare[bikeshare['start_ward'] == ward]
    total = len(ward_rides)
    if total > 0:
        classic_share = ward_rides['rideable_type_classic_bike'].mean() * 100
        electric_share = ward_rides['rideable_type_electric_bike'].mean() * 100
        # Correct calculation: sum for this ward only
        member_sum = ward_rides['member/day'].sum()
        casual_sum = ward_rides['casual/day'].sum()
        total_sum = ward_rides['total_rides/day'].sum()
        member_share = member_sum / total_sum * 100 if total_sum > 0 else 0
        casual_share = casual_sum / total_sum * 100 if total_sum > 0 else 0
    else:
        classic_share = electric_share = member_share = casual_share = 0
    customdata.append([
        out_abs, out_rel, in_abs, in_rel,
        classic_share, electric_share,
        member_share, casual_share
    ])

# Right nodes: "Ward X (To)" - use rides ending in this ward
for ward in ward_ids:
    out_abs = outgoing[ward]
    out_rel = out_abs / total_flows * 100 if total_flows else 0
    in_abs = incoming[ward]
    in_rel = in_abs / total_flows * 100 if total_flows else 0
    ward_rides = bikeshare[bikeshare['end_ward'] == ward]
    total = len(ward_rides)
    if total > 0:
        classic_share = ward_rides['rideable_type_classic_bike'].mean() * 100
        electric_share = ward_rides['rideable_type_electric_bike'].mean() * 100
        member_sum = ward_rides['member/day'].sum()
        casual_sum = ward_rides['casual/day'].sum()
        total_sum = ward_rides['total_rides/day'].sum()
        member_share = member_sum / total_sum * 100 if total_sum > 0 else 0
        casual_share = casual_sum / total_sum * 100 if total_sum > 0 else 0
    else:
        classic_share = electric_share = member_share = casual_share = 0
    customdata.append([
        out_abs, out_rel, in_abs, in_rel,
        classic_share, electric_share,
        member_share, casual_share
    ])

# --- STEP 7: Hovertemplates ---
node_hovertemplate = (
    "%{label}<br>"
    "Outgoing: %{customdata[0]} trips (%{customdata[1]:.1f}% of all trips)<br>"
    "Incoming: %{customdata[2]} trips (%{customdata[3]:.1f}% of all trips)<br>"
    "Classic Bike Share: %{customdata[4]:.1f}%<br>"
    "Electric Bike Share: %{customdata[5]:.1f}%<br>"
    # "Member Share: %{customdata[6]:.1f}%<br>"
    # "Casual Share: %{customdata[7]:.1f}%<extra></extra>"
)

# Link hover text: from -> to + value and %
link_customdata_values = [v / total_flows * 100 for _, _, v in link_customdata]
link_hover_labels = [
    f"From Ward {src} to Ward {tgt}<br>{val} trips ({val/total_flows*100:.2f}%)"
    for src, tgt, val in link_customdata
]

fig = go.Figure(data=[go.Sankey(
    arrangement="fixed",
    valueformat=".0f",
    valuesuffix=" trips",
    node=dict(
        pad=10,
        thickness=12,
        line=dict(color="black", width=0.5),
        label=labels,
        color=node_colors,
        customdata=customdata,
        hovertemplate=node_hovertemplate
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=link_colors,
        customdata=link_hover_labels,
        hovertemplate="%{customdata}<extra></extra>"
    )
)])

fig.update_layout(
    title_text="Ward-to-Ward Bike Flows",
    font_size=10,
    margin=dict(l=10, r=10, t=40, b=10),
    paper_bgcolor='rgba(0,0,0,0)'
)

fig.show()

# Save as HTML
fig.write_html("sankey_ward_bike_flows.html", full_html=True)

### Station Level Surplus and Deficit Station Mapping

In [ ]:
# import numpy as np
# import pandas as pd
# import geopandas as gpd
# from shapely.geometry import Point
# import folium

# Load bikeshare data
bikeshare = pd.read_csv("bikeshare_full2023_EDA.csv")

# Load ward polygons
wards_gdf = gpd.read_file("Wards_from_2022.geojson")
wards_gdf = wards_gdf.to_crs(epsg=4326)

# Create geometry column for each ride's start station
bikeshare['geometry'] = bikeshare.apply(lambda row: Point(row['start_lng'], row['start_lat']), axis=1)
bikeshare_gdf = gpd.GeoDataFrame(bikeshare, geometry='geometry', crs="EPSG:4326")

# Spatial join: keep only rides where the station is within a ward polygon
bikeshare_in_wards = gpd.sjoin(bikeshare_gdf, wards_gdf, how='inner', predicate='within')

# Calculate the mean coordinates for each station (single representative point)
station_coords = (
    bikeshare_in_wards.groupby('start_station_id')[['start_lat', 'start_lng']]
    .mean()
    .reset_index()
)

# Prepare the final dataframe with required columns, including 'rideable_type' and 'end_station_id'
df = bikeshare_in_wards[['start_station_id', 'end_station_id', 'date', 'is_weekend', 'hour', 'rideable_type']].copy()
df = df.merge(station_coords, on='start_station_id', how='left')

# Reorder columns as requested
df = df[['start_station_id', 'end_station_id', 'start_lat', 'start_lng', 'date', 'is_weekend', 'hour', 'rideable_type']]

# Preview
df.head()

In [ ]:
# import pandas as pd
# import geopandas as gpd
# from shapely.geometry import Point
# import folium

# 1. Assign each station to a ward
stations = df[['start_station_id', 'start_lat', 'start_lng']].drop_duplicates()
stations['geometry'] = stations.apply(lambda row: Point(row['start_lng'], row['start_lat']), axis=1)
stations_gdf = gpd.GeoDataFrame(stations, geometry='geometry', crs="EPSG:4326")
stations_gdf = gpd.sjoin(stations_gdf, wards_gdf[['geometry']], how='left', predicate='within')
stations_gdf = stations_gdf.rename(columns={'index_right': 'ward_index'})

# 2. Calculate mean weekday departures (all hours) per station
departures_df = df[df['is_weekend'] == False]
departures_day = (
    departures_df.groupby(['start_station_id', 'date'])
    .size()
    .reset_index(name='departures_per_day')
)
departures_agg = (
    departures_day.groupby('start_station_id')['departures_per_day']
    .mean()
    .reset_index(name='mean_departures')
)

# 3. Calculate mean weekday arrivals (all hours) per station
arrivals_df = bikeshare_in_wards[bikeshare_in_wards['is_weekend'] == False]
arrivals_day = (
    arrivals_df.groupby(['end_station_id', 'date'])
    .size()
    .reset_index(name='arrivals_per_day')
)
arrivals_agg = (
    arrivals_day.groupby('end_station_id')['arrivals_per_day']
    .mean()
    .reset_index(name='mean_arrivals')
)

# 4. Merge departures and arrivals to get net flow per station
station_flows = stations_gdf.merge(departures_agg, left_on='start_station_id', right_on='start_station_id', how='left')
station_flows = station_flows.merge(arrivals_agg, left_on='start_station_id', right_on='end_station_id', how='left')
station_flows['mean_departures'] = station_flows['mean_departures'].fillna(0)
station_flows['mean_arrivals'] = station_flows['mean_arrivals'].fillna(0)
station_flows['net_flow'] = station_flows['mean_arrivals'] - station_flows['mean_departures']

# 5. Prepare wards_gdf for folium (only geometry column to avoid serialization issues)
wards_gdf_simple = wards_gdf[['geometry']].copy()

# 6. Visualize on a map for the whole city
m = folium.Map(
    location=[station_flows['start_lat'].mean(), station_flows['start_lng'].mean()],
    zoom_start=12,
    tiles="CartoDB positron"
)

# Add all ward polygons
folium.GeoJson(
    wards_gdf_simple,
    style_function=lambda feature: {
        'fillOpacity': 0.05,
        'color': 'black',
        'weight': 2
    }
).add_to(m)

# Add stations with color/size by net flow
for _, row in station_flows.iterrows():
    color = 'blue' if row['net_flow'] > 0 else 'red'
    folium.CircleMarker(
        location=[row['start_lat'], row['start_lng']],
        radius=5 + abs(row['net_flow'])**0.5,  # size by magnitude
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Station ID:</b> {row['start_station_id']}<br>"
            f"<b>Net Flow (all day):</b> {row['net_flow']:.2f}<br>"
            f"<b>Arrivals (all day):</b> {row['mean_arrivals']:.2f}<br>"
            f"<b>Departures (all day):</b> {row['mean_departures']:.2f}",
            max_width=250
        )
    ).add_to(m)

m.save('citywide_net_bikeflow_allday.html')
m

In [ ]:
# Filter for stations with net flow > 1 or < -1
filtered_flows = station_flows[(station_flows['net_flow'] > 1) | (station_flows['net_flow'] < -1)]

# 5. Prepare wards_gdf for folium (only geometry column to avoid serialization issues)
wards_gdf_simple = wards_gdf[['geometry']].copy()

# 6. Visualize on a map for the whole city
m = folium.Map(
    location=[filtered_flows['start_lat'].mean(), filtered_flows['start_lng'].mean()],
    zoom_start=12,
    tiles="CartoDB positron"
)

# Add all ward polygons
folium.GeoJson(
    wards_gdf_simple,
    style_function=lambda feature: {
        'fillOpacity': 0.05,
        'color': 'black',
        'weight': 2
    }
).add_to(m)

# Add filtered stations with color/size by net flow
for _, row in filtered_flows.iterrows():
    color = 'blue' if row['net_flow'] > 0 else 'red'
    folium.CircleMarker(
        location=[row['start_lat'], row['start_lng']],
        radius=5 + abs(row['net_flow'])**0.5,  # size by magnitude
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Station ID:</b> {row['start_station_id']}<br>"
            f"<b>Net Flow (all day):</b> {row['net_flow']:.2f}<br>"
            f"<b>Arrivals (all day):</b> {row['mean_arrivals']:.2f}<br>"
            f"<b>Departures (all day):</b> {row['mean_departures']:.2f}",
            max_width=250
        )
    ).add_to(m)

m.save('citywide_net_bikeflow_allday_filtered.html')
m

## Final Data Frame for Bike (Re-)Distribution Optimization

The station ID's grouped in this csv can be entered into any route calculator to generate and visualize the optimal bike redistribution route.

In [ ]:
import numpy as np
import pandas as pd

def haversine_matrix(lat1, lon1, lat2, lon2):
    """
    Vectorized haversine distance between all pairs of (lat1, lon1) and (lat2, lon2).
    Returns a (len(lat1), len(lat2)) matrix.
    """
    R = 6371
    lat1 = np.radians(np.asarray(lat1))[:, None]
    lon1 = np.radians(np.asarray(lon1))[:, None]
    lat2 = np.radians(np.asarray(lat2))[None, :]
    lon2 = np.radians(np.asarray(lon2))[None, :]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

valid_types = ['classic_bike', 'electric_bike']
threshold = 1
all_pairings = []

for bike_type in valid_types:
    surplus = station_flows[
        (station_flows['rideable_type'] == bike_type) & (station_flows['net_flow'] > threshold)
    ].copy()
    deficit = station_flows[
        (station_flows['rideable_type'] == bike_type) & (station_flows['net_flow'] < -threshold)
    ].copy()
    if surplus.empty or deficit.empty:
        continue

    # Prepare arrays for vectorized matching
    surplus_ids = surplus['start_station_id'].values
    surplus_flows = surplus['net_flow'].values.copy()
    surplus_lats = surplus['start_lat'].values
    surplus_lngs = surplus['start_lng'].values

    deficit_ids = deficit['start_station_id'].values
    deficit_flows = deficit['net_flow'].values.copy()
    deficit_lats = deficit['start_lat'].values
    deficit_lngs = deficit['start_lng'].values

    # Compute distance matrix (surplus x deficit)
    dist_matrix = haversine_matrix(surplus_lats, surplus_lngs, deficit_lats, deficit_lngs)

    # Greedy vectorized matching
    pairings = []
    while (surplus_flows > threshold).any() and (deficit_flows < -threshold).any():
        # Find the largest surplus
        s_idx = np.argmax(surplus_flows)
        # Only consider deficits still < -threshold
        valid_deficits = np.where(deficit_flows < -threshold)[0]
        if valid_deficits.size == 0:
            break
        # Find nearest valid deficit
        dists = dist_matrix[s_idx, valid_deficits]
        d_idx = valid_deficits[np.argmin(dists)]
        move_bikes = min(surplus_flows[s_idx], abs(deficit_flows[d_idx]))
        pairings.append({
            'rideable_type': bike_type,
            'from_station': surplus_ids[s_idx],
            'to_station': deficit_ids[d_idx],
            'bikes_to_move': int(round(move_bikes)),
            'distance_km': float(dist_matrix[s_idx, d_idx]),
            'from_surplus': round(surplus_flows[s_idx], 2),
            'to_deficit': round(deficit_flows[d_idx], 2)
        })
        # Update flows
        surplus_flows[s_idx] -= move_bikes
        deficit_flows[d_idx] += move_bikes

    pairings_distance_ridabletype_df = pd.DataFrame(pairings)
    all_pairings.append(pairings_distance_ridabletype_df)

pairings_distance_ridabletype_df = pd.concat(all_pairings, ignore_index=True)
print(pairings_distance_ridabletype_df)